In [62]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/House_Prices.csv")

In [63]:
df["TotalArea"] = df[
    ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]
].sum(axis=1, skipna=True)

In [64]:
df["TotalBathrooms"] = (
    df["FullBath"].fillna(0)
    + 0.5 * df["HalfBath"].fillna(0)
    + df["BsmtFullBath"].fillna(0)
    + 0.5 * df["BsmtHalfBath"].fillna(0)
)

TotalBathrooms combines full and half bathrooms from both above-ground and basement levels into one feature. Half bathrooms are weighted as 0.5.

In [65]:
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

HouseAge represents how old the house was when it was sold. It is calculated as YrSold - YearBuilt.

In [66]:
df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]

YearsSinceRemodel represents how many years passed between the last renovation and the sale of the house.

In [67]:
df.loc[df["HouseAge"] < 0, "HouseAge"] = np.nan
df.loc[df["YearsSinceRemodel"] < 0, "YearsSinceRemodel"] = np.nan

In [68]:
df["TotalPorchArea"] = (
    df["WoodDeckSF"]
    + df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
)

TotalPorchArea combines the main porch and deck surface areas into one feature representing the property's total outdoor living area.

checking new features

In [69]:
engineered_features = [
    "TotalArea",
    "TotalBathrooms",
    "HouseAge",
    "YearsSinceRemodel",
    "TotalPorchArea"
]

df[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
TotalArea,2919.0,2547.482700,805.120840,334.0,2000.0,2448.0,2991.50,11752.0
TotalBathrooms,2919.0,2.218397,0.808840,1.0,1.5,2.0,2.50,7.0
HouseAge,2918.0,36.492803,30.333442,0.0,7.0,35.0,54.75,136.0
YearsSinceRemodel,2916.0,23.553841,20.887566,0.0,4.0,15.0,43.00,60.0
TotalPorchArea,2919.0,182.959575,160.021404,0.0,48.0,164.0,266.50,1424.0


In [70]:
df[engineered_features].isna().sum()

TotalArea            0
TotalBathrooms       0
HouseAge             1
YearsSinceRemodel    3
TotalPorchArea       0
dtype: int64

In [71]:
for col in engineered_features:
    print(col, (df[col] < 0).sum())

TotalArea 0
TotalBathrooms 0
HouseAge 0
YearsSinceRemodel 0
TotalPorchArea 0


integrate the engineered features into the preprocessing flow

In [76]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [78]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

In [80]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [81]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [82]:
print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

Train shape: (2335, 288)
Test shape: (584, 288)


In [83]:
print("Engineered features:")
print(engineered_features)

Engineered features:
['TotalArea', 'TotalBathrooms', 'HouseAge', 'YearsSinceRemodel', 'TotalPorchArea']
